# Radiomics from ovarian ultrasound: a research demonstration

This notebook walks through MMOTU OTU_2d from data audit to a leakage-aware model comparison. It is intended for a clinical research discussion, not diagnosis. It uses the official eight-category task and does not create a benign/malignant mapping.

## 1. Research question and dataset

Can a conservative set of measurements from expert-annotated image regions distinguish the dataset's official image categories? The repository describes 1,469 2D ultrasound images and provides global category labels and pixel annotations. The official lists identify images by filename; no reliable patient identifier is used here.

If the public files are not present, the code falls back to clearly marked synthetic observations so that the software demonstration still runs.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, HTML
from src.data import load_mmotu
from src.experiment import run_experiment


## 2. Inspect the dataset before modeling

The audit checks the official label lists, image and mask availability, class distribution, dimensions, empty masks, filename duplication, and whether patient grouping is available. The complete audit is saved as JSON.

In [2]:
data_root = Path('data/raw/OTU_2d')
try:
    labels_preview, audit_preview = load_mmotu(data_root)
    audit_summary = {
        'dataset': audit_preview['dataset'],
        'labeled rows': audit_preview['n_label_rows'],
        'usable rows': audit_preview['usable_labeled_rows'],
        'excluded rows': audit_preview['excluded_rows_from_modeling'],
        'images found': audit_preview['n_images_found'],
        'masks found': audit_preview['n_masks_found'],
        'rows missing masks': len(audit_preview['missing_masks_for_labeled_rows']),
        'class distribution': audit_preview['class_distribution'],
        'patient grouping available': audit_preview['patient_grouping_available'],
    }
except (FileNotFoundError, ValueError) as exc:
    audit_summary = {'dataset': 'synthetic fallback will be used', 'reason': str(exc)}
display(pd.Series(audit_summary, dtype='object').to_frame('audit value'))

,audit value
dataset,MMOTU OTU_2d
labeled rows,1469
usable rows,1269
excluded rows,200
images found,1469
masks found,1269
rows missing masks,200
class distribution,"{'chocolate cyst': 336, 'serous cystadenoma': ..."
patient grouping available,False


## 3. Extract radiomic features

The mask defines the region of interest. The default extractor is PyRadiomics and computes first-order, GLCM, GLRLM, and 2D shape features. If PyRadiomics is unavailable, the explicitly labeled lightweight NumPy fallback keeps the demo runnable. Physical pixel spacing is unavailable, so shape lengths and areas use pixel units and have no physical interpretation. The selected engine, version, settings, and feature names are recorded in the outputs.

## 4. Leakage-safe validation and model comparison

The original stratified image-level CV is rerun as a baseline. The primary corrected estimate uses duplicate-grouped image-level cross-validation: all images with the same decoded-pixel group stay together. The official validation set is separately cleaned of pixel groups also present in training. Imputation, variance filtering, scaling, feature selection, and model fitting remain inside each CV training fold. No patient grouping is claimed.

In [3]:
result = run_experiment(data_root=data_root, output_dir=Path('outputs'), seed=42, n_splits=5)
audit = result['audit']
display(pd.DataFrame([{'model': name, 'duplicate-grouped CV macro F1': f"{metrics['macro_f1']['mean']:.3f} ± {metrics['macro_f1']['std']:.3f}", 'balanced accuracy': f"{metrics['balanced_accuracy']['mean']:.3f} ± {metrics['balanced_accuracy']['std']:.3f}", 'train-validation accuracy gap': f"{metrics['accuracy_gap_mean']:.3f}"} for name, metrics in result['cv_results'].items()]))
display(pd.Series({
    'engine': result['feature_info']['engine'],
    'version': result['feature_info']['pyradiomics_version'],
    'feature count': len(result['feature_info']['feature_names']),
    'spacing': result['feature_info']['settings']['spacing'],
}))

Dataset: MMOTU OTU_2d | labeled rows: 1469 | features: 67
CV folds: 5 | duplicate groups: 617 | cross-fold duplicate clusters: 0 | report: outputs/demo_report.html


,model,duplicate-grouped CV macro F1,balanced accuracy,train-validation accuracy gap
0,Dummy,0.052 ± 0.001,0.125 ± 0.000,-0.000
1,Logistic Regression,0.364 ± 0.045,0.395 ± 0.050,0.025
2,Random Forest,0.402 ± 0.019,0.397 ± 0.017,0.424
3,XGBoost,0.330 ± 0.045,0.334 ± 0.031,0.256


engine           PyRadiomics firstorder + GLCM + GLRLM + 2D shape
version                                    3.1.0a2.post8+g6a761c4
feature count                                                  67
spacing                 pixel units; physical spacing unavailable
dtype: object

## 5. Why duplicate-safe validation matters

Source folders contain duplicate file copies, but the loader contributes one observation per labeled image. Exact decoded-pixel grouping checks both the physical source inventory and the actual model rows. The cells below show an example duplicate pair, whether any modeled duplicate crossed the original folds or official partitions, before/after CV metrics, and the duplicate-clean official validation results.

In [4]:
duplicate_audit = json.loads(Path('outputs/duplicate_leakage_audit.json').read_text())
duplicate_rows = pd.read_csv('outputs/duplicate_groups.csv')
repeated_groups = duplicate_rows.groupby('duplicate_group_id').filter(lambda group: len(group) > 1)
if not repeated_groups.empty:
    example_id = repeated_groups.groupby('duplicate_group_id').size().sort_values(ascending=False).index[0]
    display(repeated_groups.loc[repeated_groups.duplicate_group_id == example_id,
                                ['duplicate_group_id', 'filename', 'label_name', 'source_partition', 'source_folder']].head(6))
else:
    display(pd.DataFrame({'duplicate example': ['No repeated pixel groups found']}))
display(pd.Series({key: duplicate_audit[key] for key in [
    'n_usable_images', 'n_unique_pixel_groups', 'n_duplicate_groups',
    'n_images_in_duplicate_groups', 'n_duplicate_groups_spanning_original_cv_folds',
    'n_duplicate_groups_spanning_official_train_validation',
    'original_validation_image_count', 'contaminated_validation_images_removed',
    'cleaned_validation_image_count'
]}, name='duplicate audit'))
comparison = pd.read_csv('outputs/duplicate_safe_model_comparison.csv')
comparison_columns = [
    'model', 'original_cv_accuracy_mean', 'duplicate_safe_cv_accuracy_mean',
    'original_cv_macro_f1_mean', 'duplicate_safe_cv_macro_f1_mean',
    'original_cv_balanced_accuracy_mean', 'duplicate_safe_cv_balanced_accuracy_mean',
    'original_cv_macro_ovr_auc_mean', 'duplicate_safe_cv_macro_ovr_auc_mean',
    'original_cv_train_accuracy_mean', 'duplicate_safe_cv_train_accuracy_mean',
    'original_cv_train_validation_gap_mean', 'duplicate_safe_cv_train_validation_gap_mean',
    'original_official_validation_macro_f1', 'duplicate_clean_official_validation_macro_f1'
]
display(comparison[comparison_columns].round(3))
from src.duplicate_analysis import leakage_conclusion
display(HTML(f"<p><strong>Interpretation:</strong> {leakage_conclusion(comparison, groups_crossing_original_folds=duplicate_audit['n_duplicate_groups_spanning_original_cv_folds'])}</p>"))

,duplicate_group_id,filename,label_name,source_partition,source_folder
978,pixel_0069ae7529cbb4963299ad7689b0caba63c0f1b2...,194.JPG,chocolate cyst,train,OTU_2d/images
979,pixel_0069ae7529cbb4963299ad7689b0caba63c0f1b2...,194.JPG,chocolate cyst,train,figshare_mmotu/train/train_image


n_usable_images                                          1269
n_unique_pixel_groups                                    1269
n_duplicate_groups                                        617
n_images_in_duplicate_groups                             1234
n_duplicate_groups_spanning_original_cv_folds               0
n_duplicate_groups_spanning_official_train_validation       0
original_validation_image_count                           402
contaminated_validation_images_removed                      0
cleaned_validation_image_count                            402
Name: duplicate audit, dtype: int64

,model,original_cv_accuracy_mean,duplicate_safe_cv_accuracy_mean,original_cv_macro_f1_mean,duplicate_safe_cv_macro_f1_mean,original_cv_balanced_accuracy_mean,duplicate_safe_cv_balanced_accuracy_mean,original_cv_macro_ovr_auc_mean,duplicate_safe_cv_macro_ovr_auc_mean,original_cv_train_accuracy_mean,duplicate_safe_cv_train_accuracy_mean,original_cv_train_validation_gap_mean,duplicate_safe_cv_train_validation_gap_mean,original_official_validation_macro_f1,duplicate_clean_official_validation_macro_f1
0,Dummy,0.263,0.263,0.052,0.052,0.125,0.125,0.500,0.500,0.263,0.263,0.000,-0.000,0.053,0.053
1,Logistic Regression,0.478,0.480,0.367,0.364,0.396,0.395,0.778,0.786,0.504,0.505,0.027,0.025,0.366,0.366
2,Random Forest,0.549,0.552,0.394,0.402,0.388,0.397,0.801,0.812,0.976,0.977,0.427,0.424,0.374,0.374
3,XGBoost,0.535,0.537,0.314,0.330,0.322,0.334,0.804,0.806,0.787,0.794,0.252,0.256,0.333,0.333


## 6. Confusion matrices and calibration

The HTML report contains confusion matrices and one-vs-rest calibration plots for the original official validation set. Calibration describes how predicted probabilities compare with observed class frequencies; it does not establish clinical usefulness.

In [5]:
report_path = Path('outputs/demo_report.html')
display(HTML(f"<p>Report generated: <a href='{report_path}'>{report_path}</a></p>"))
display(pd.DataFrame(result['features']).describe().T.head(12))


,count,mean,std,min,25%,50%,75%,max
original_firstorder_10Percentile,867.0,2.735433e+01,2.514218e+01,1.000000e+00,7.000000e+00,2.100000e+01,4.100000e+01,2.030000e+02
original_firstorder_90Percentile,867.0,1.027915e+02,4.420377e+01,1.300000e+01,6.800000e+01,1.000000e+02,1.320000e+02,2.440000e+02
original_firstorder_Energy,867.0,3.515526e+08,4.152317e+08,5.128381e+06,9.502172e+07,2.028569e+08,4.471483e+08,3.580060e+09
original_firstorder_Entropy,867.0,2.612316e+00,6.177109e-01,5.656499e-01,2.207973e+00,2.677871e+00,3.116075e+00,3.874270e+00
original_firstorder_InterquartileRange,867.0,3.871251e+01,2.195855e+01,1.000000e+00,2.300000e+01,3.500000e+01,5.000000e+01,1.580000e+02
original_firstorder_Kurtosis,867.0,1.038959e+01,1.095616e+01,1.609119e+00,3.698338e+00,6.572958e+00,1.366842e+01,1.486270e+02
original_firstorder_Maximum,867.0,2.454994e+02,1.744429e+01,1.040000e+02,2.430000e+02,2.520000e+02,2.550000e+02,2.550000e+02
original_firstorder_Mean,867.0,6.103577e+01,3.286012e+01,5.478070e+00,3.577160e+01,5.544866e+01,8.063476e+01,2.236453e+02
original_firstorder_MeanAbsoluteDeviation,867.0,2.513872e+01,1.032529e+01,5.725738e+00,1.750853e+01,2.334097e+01,3.073350e+01,7.539760e+01
original_firstorder_Median,867.0,5.437140e+01,3.632246e+01,1.000000e+00,2.600000e+01,4.800000e+01,7.600000e+01,2.260000e+02


## 7. Methodological safeguards and limitations

- Patient grouping is unavailable; this audit groups exact duplicate images only.
- Every `duplicate_group_id` stays within one validation fold in duplicate-grouped CV.
- Validation images sharing a duplicate group with training are removed from the duplicate-clean evaluation.
- Feature selection, imputation, and scaling are trained within each fold.
- No clinical covariates, patient IDs, benign/malignant labels, or physical spacing are fabricated.
- The dataset is small and comes from a single source; acquisition and annotation differences may affect radiomic features.
- This experiment does not prove clinical validity, diagnostic utility, external generalization, or patient-level performance.

Review `outputs/data_audit.json`, `outputs/duplicate_groups.csv`, `outputs/duplicate_leakage_audit.json`, `outputs/duplicate_safe_model_comparison.csv`, `outputs/duplicate_safe_validation.json`, `outputs/radiomics_config.json`, `outputs/selected_features_by_fold.csv`, `outputs/metrics.json`, and `outputs/demo_report.html` for the persisted audit trail.